# Lesson 3.4: Debug a looping agent from its trace
*Build an AI Agent from Scratch with LangGraph · Chapter 3*

**Goal:** find the loops in agent runs by turning what you see in one trace into rules a program can check.

**What you will be able to do**
- read a trace as a story: errors, repeats, the fix, and what came after it
- write a detector for three loop patterns: (a) identical repeats, (b) green but no finish, (c) work after the fix
- count the patterns over all 180 baseline runs, and measure a fix on the same 30 runs as lesson 3.2
- treat a flag as a reason to open a trace, not as a verdict

**How this notebook runs.** Every run is a replay from `nightwatch/runs/cache` (`mode="replay"`). No cell calls the API.
Python counts trace entries from 0; the slides number the steps 1 to 14.

In [1]:
# Setup: find the course repo's nightwatch/ folder, load .env, check the key.
import os, sys
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

for cand in ["../../nightwatch", "../../../nightwatch", "nightwatch"]:   # relative to this notebook
    if (Path(cand) / "agent_v0.py").exists():
        NW = Path(cand).resolve()
        break
else:
    raise SystemExit("Can't find nightwatch/. Open this notebook from the course repo (lessons/<lesson>/notebook/).")
sys.path.insert(0, str(NW))

load_dotenv(find_dotenv(usecwd=True))          # the .env at the repo root, if there is one
HAS_KEY = bool(os.environ.get("GROQ_API_KEY"))
print("nightwatch/ found:", (NW / "agent_v0.py").exists(), "| Python", sys.version.split()[0])
if HAS_KEY:
    print("GROQ_API_KEY loaded.")
else:
    print("No GROQ_API_KEY. Get a free key as in lesson 0.2 (console.groq.com -> API Keys) and put")
    print("GROQ_API_KEY=... in the .env file at the repo root.")
print("This notebook has no cell that calls the API: every run is a replay from nightwatch/runs/cache.")

nightwatch/ found: True | Python 3.13.16
GROQ_API_KEY loaded.
This notebook has no cell that calls the API: every run is a replay from nightwatch/runs/cache.


## 1. Recall: the waterfall (lesson 3.3)

In the 7-call run of 3.3 the model took 2,097 ms and the tools 0.05 ms. Today we ask the trace a new question:
not how long each step took, but **what it did**. That run's path:

In [2]:
import json
from collections import Counter
import agent_v0
from brightcart.incidents import generate
from brightcart.sim import World

incs = {x.id: x for x in generate()}
r = agent_v0.run(incs["upstream_deploy-0"], mode="replay", sample=0)
print(" -> ".join(t["tool"] for t in r["trace"]))

check_health -> read_logs -> read_logs -> recent_deploys -> rollback_deploy -> check_health -> finish


## 2. Key words

| Word | Meaning |
|---|---|
| trace | the steps of one run: tool, arguments, result |
| loop pattern | a shape of wasted steps that keeps coming back |
| detector | a function that reads a trace and returns flags |
| flag | a pattern showed up: a symptom, not a verdict |
| hidden fix | the write that solves the incident; the grader knows it, the agent does not |
| exact replay | re-running a recorded run from the cache, step for step |

## 3. The idea, then a hand trace

Incident `stuck_consumer-0`, sample 2. Alert: *queue email_outbox depth above 1000*; hidden fix:
`restart_service(email-worker)`.

| entry | tool (args) | result |
|---|---|---|
| 0 | `queue_depth(service=email_outbox)` | email_outbox: 7734 |
| 1–3 | read_logs / query_db / service_status on `email_outbox` | three ERRORs (a queue is not a service; a missing column) |
| 4 | `read_logs(email-worker, n=20)` | normal INFO lines |
| 5–8 | runbook (ERROR), deploys `[]`, config changes `[]`, status | nothing new |
| 9 | `read_logs(email-worker, n=50)` | the same lines |
| 10–12 | `read_logs(email-worker, n=200)` **three times** | identical |
| 13 | `restart_service(email-worker)` | the right fix, at the last allowed step |
| stop | step limit | healthy True, no health check, no finish: a fail |

In our simulated shop nothing changes without a write, so entries 11 and 12 cannot say anything new.

**Four rules over a trace**
- (a) writes: the same tool with the same arguments twice in one run;
- (a) reads: the same read again **with no write in between**;
- (b) a health check came back green, but the run never called `finish`;
- (c) more than two entries after the right fix (the ideal tail is fix, `check_health`, `finish`).

**By hand:** each read gets the key (tool, arguments, writes so far). `(read_logs, {n:200, email-worker}, 0)` appears 3 times,
so (a) fires. (b) stays quiet: no health check at all. (c) stays quiet: the fix is the last entry.

## 4. Write it yourself

First: which baseline runs can we replay **exactly** (same steps, same writes)? The cache keeps only what was recorded.

In [3]:
rows = [json.loads(line) for line in open(NW / "runs" / "baseline_gpt-oss-20b.jsonl", encoding="utf-8")]
traces = {}
for row in rows:
    try:
        a = agent_v0.run(incs[row["incident"]], mode="replay", sample=row["sample"])
    except LookupError:                       # a reply that was never saved
        continue
    if a["steps"] == row["steps"] and a["writes"] == [tuple(w) for w in row["writes"]]:
        traces[(row["incident"], row["sample"])] = a
print(f"exact replays: {len(traces)} of {len(rows)}")

run = traces[("stuck_consumer-0", 2)]
for i, t in enumerate(run["trace"]):
    print(f"{i:2d} {t.get('tool') or t.get('error'):16} {json.dumps(t.get('args'))[:40]:40} -> {str(t.get('result', ''))[:40]}")
print(f"stop: {run['stop_reason']} | final {run['final']} | healthy {run['healthy']} | writes {run['writes']}")

exact replays: 39 of 180
 0 queue_depth      {"service": "email_outbox"}              -> {'email_outbox': 7734}
 1 read_logs        {"n": 20, "service": "email_outbox"}     -> ERROR: "unknown service 'email_outbox'"
 2 query_db         {"sql": "SELECT name, state, replicas FR -> ERROR: no such column: state
 3 service_status   {"service": "email_outbox"}              -> ERROR: "unknown service 'email_outbox'; 
 4 read_logs        {"n": 20, "service": "email-worker"}     -> ['03:16 INFO email-worker handled reques
 5 read_runbook     {"topic": "stuck_consumer"}              -> ERROR: "no runbook 'stuck_consumer'; top
 6 recent_deploys   {}                                       -> []
 7 config_changes   {}                                       -> []
 8 service_status   {"service": "email-worker"}              -> {'email-worker': {'version': 'v2.4.2', '
 9 read_logs        {"n": 50, "service": "email-worker"}     -> ['03:16 INFO email-worker handled reques
10 read_logs        {"n": 200, "

In [4]:
def fix_step(trace, label):
    """Index of the trace entry where the labelled fix was applied (None if never)."""
    if not label["fix"]:
        return None
    for i, t in enumerate(trace):
        args = t.get("args") or {}
        target = args.get("service") or args.get("flag")
        arg = args.get("replicas") or args.get("value") or (f"{args['key']}={args['value']}" if "key" in args else None)
        if t.get("tool") in agent_v0.WRITE_TOOLS and World._matches(label["fix"], (t["tool"], target, arg)):
            return i
    return None


def detect(run):
    trace, flags = run["trace"], []
    seen, writes = Counter(), 0          # a read repeated with no write in between can only return the same answer
    for t in trace:
        if t.get("tool"):
            is_write = t["tool"] in agent_v0.WRITE_TOOLS
            seen[(t["tool"], json.dumps(t.get("args"), sort_keys=True), None if is_write else writes)] += 1
            writes += is_write
    for (tool, args, _), n in seen.items():
        if n > 1:
            kind = "WRITE" if tool in agent_v0.WRITE_TOOLS else "read"
            flags.append(f"(a) identical {kind} x{n}: {tool}{args}")
    green = [i for i, t in enumerate(trace) if t.get("tool") == "check_health" and "'healthy': True" in t.get("result", "")]
    if green and run["final"] is None:
        flags.append(f"(b) health green at entry {green[0]} but no finish()")
    fx = fix_step(trace, run["label"])
    if fx is not None and len(trace) - 1 - fx > 2:     # fix, check_health, finish is the ideal tail
        flags.append(f"(c) {len(trace) - 1 - fx} entries after the fix at entry {fx}")
    return flags


print(detect(run))

['(a) identical read x3: read_logs{"n": 200, "service": "email-worker"}']


Note the `None` for writes in the key: the same write repeats even with other writes in between, while a read
after a change counts as new.

## 5. Check against the lesson's numbers

No library ships this detector; tracing tools give you the traces, the rules are yours. So the check is the
lesson's own result: on the 39 exact traces the flags must be (a) WRITE 2, (a) read 6, (b) 3, (c) 6.

In [5]:
cnt = Counter()
for (iid, s), a in sorted(traces.items()):
    fl = detect(a)
    for f in fl:
        cnt[f[:3] + (" WRITE" if "WRITE" in f else " read" if f.startswith("(a)") else "")] += 1
    if fl:
        print(f"{iid:17} s{s} {'PASS' if agent_v0.grade(a)['pass'] else 'fail'}  " + " | ".join(f[:60] for f in fl))
print(dict(cnt), "| matches the lesson:", dict(cnt) == {"(a) WRITE": 2, "(c)": 6, "(a) read": 6, "(b)": 3})

bad_deploy-2      s0 PASS  (a) identical WRITE x2: rollback_deploy{"service": "web"} | (c) 5 entries after the fix at entry 3
bad_deploy-3      s1 PASS  (c) 3 entries after the fix at entry 4
bad_deploy-5      s0 PASS  (c) 3 entries after the fix at entry 3
false_alarm-5     s0 PASS  (a) identical read x2: check_health{}
memory_leak-5     s2 fail  (b) health green at entry 12 but no finish() | (c) 3 entries after the fix at entry 10
stuck_consumer-0  s2 fail  (a) identical read x3: read_logs{"n": 200, "service": "email
stuck_consumer-2  s0 fail  (a) identical read x3: read_logs{"n": 200, "service": "email
stuck_consumer-3  s0 PASS  (c) 3 entries after the fix at entry 8
stuck_consumer-3  s2 PASS  (c) 3 entries after the fix at entry 9
stuck_consumer-5  s1 fail  (a) identical read x2: read_logs{"n": 50, "service": "email- | (a) identical read x2: read_logs{"n": 200, "service": "email
traffic_spike-0   s0 fail  (a) identical read x2: recent_deploys{} | (a) identical WRITE x2: scale_servi

Over **all 180** runs we have no full traces, only what each run saved (writes, final, end health). Count from that:

In [6]:
rep_write = [r for r in rows if len({json.dumps(w) for w in r["writes"]}) < len(r["writes"])]
nofin = [r for r in rows if not r["final"]]
green_nofin = [r for r in nofin if r["healthy"]]
after_fix = []
for r in rows:
    lab = incs[r["incident"]].label
    if lab["fix"]:
        idx = [i for i, w in enumerate(r["writes"]) if World._matches(lab["fix"], tuple(w))]
        if idx and idx[0] < len(r["writes"]) - 1:
            after_fix.append(r)
print(f"(a) identical repeated write: {len(rep_write)}")
print(f"(b) no finish but healthy at the end: {len(green_nofin)} of {len(nofin)}, passed {sum(r['pass'] for r in green_nofin)}")
print(f"(c) wrote again after the right fix: {len(after_fix)}, still passed {sum(r['pass'] for r in after_fix)}")

(a) identical repeated write: 44
(b) no finish but healthy at the end: 31 of 46, passed 0
(c) wrote again after the right fix: 45, still passed 26


## 6. Where it shows up in agents

Run the detector over every batch of eval runs, next to the score: the score says which runs failed, the flags say
which traces to open first. Here: the same 30 runs as lesson 3.2, before and after its guards.

In [7]:
def counts(rs):
    out = Counter()
    for r in rs:
        lab = incs[r["incident"]].label
        ws = [tuple(w) for w in r["writes"]]
        out["(a) repeated write"] += len(set(ws)) < len(ws)
        out["(b) healthy, no finish"] += (not r["final"]) and r["healthy"]
        idx = [i for i, w in enumerate(ws) if lab["fix"] and World._matches(lab["fix"], w)]
        out["(c) write after fix"] += bool(idx) and idx[0] < len(ws) - 1
        out["no finish"] += not r["final"]
    return dict(out)


worst = ["pool_config-3", "stuck_consumer-1", "stuck_consumer-5", "traffic_spike-1", "memory_leak-0",
         "memory_leak-1", "memory_leak-2", "memory_leak-4", "memory_leak-5", "stuck_consumer-0"]
before = [r for r in rows if r["incident"] in worst]
after = [agent_v0.run(incs[r["incident"]], mode="replay", sample=r["sample"], guards=True, token_budget=60000)
         for r in before]
print("before:", counts(before))
print("after :", counts(after))

before: {'(a) repeated write': 7, '(b) healthy, no finish': 15, '(c) write after fix': 9, 'no finish': 24}
after : {'(a) repeated write': 0, '(b) healthy, no finish': 10, '(c) write after fix': 4, 'no finish': 14}


Thirty runs is a small sample: a direction, not precise rates.

## 7. Traps

**Trap 1: reading only the pass rate.** Passing runs can still hide waste.

In [8]:
print(f"{sum(r['pass'] for r in after_fix)} of {len(after_fix)} runs that wrote again after the fix still PASSED")

26 of 45 runs that wrote again after the fix still PASSED


**Trap 2: treating every flag as a bug.** A flag says where to look; then you read the trace.

In [9]:
fa = traces[("false_alarm-5", 0)]
print("false_alarm-5 s0:", detect(fa), "| grade:", agent_v0.grade(fa)["pass"])
print("two health checks on a false alarm are harmless: the run passed")

false_alarm-5 s0: ['(a) identical read x2: check_health{}'] | grade: True
two health checks on a false alarm are harmless: the run passed


**Trap 3: blocking every repeated read.** Live logs do change. Flag repeated reads; block only identical writes.
The writes-so-far part of the key keeps a read *after a change* from counting as a repeat:

In [10]:
fake = {"label": {"fix": None}, "final": {"x": 1}, "trace": [
    {"tool": "read_logs", "args": {"service": "web"}, "result": "a"},
    {"tool": "restart_service", "args": {"service": "web"}, "result": "done"},
    {"tool": "read_logs", "args": {"service": "web"}, "result": "b"}]}
print("read, write, same read ->", detect(fake) or "(nothing flagged)")
fake["trace"].pop(1)
print("read, same read        ->", detect(fake))

read, write, same read -> (nothing flagged)
read, same read        -> ['(a) identical read x2: read_logs{"service": "web"}']


**Trap 4: debugging a replay that is not exact.** Only 39 of 180 replay step for step; the rest hit a reply that was never saved.

In [11]:
try:
    agent_v0.run(incs["bad_deploy-4"], mode="replay", sample=1)
    print("replayed")
except LookupError as e:
    print("bad_deploy-4 s1:", type(e).__name__, str(e)[:40])
print("exact replay available:", ("bad_deploy-4", 1) in traces)

bad_deploy-4 s1: LookupError not in cache: f94fd198220eea4fda4a0b40
exact replay available: False


## 8. Exercises (no API: every check replays from the cache)

In [12]:
def check(n, test):
    """Run the asserts in test(). Print one clear line instead of stopping the notebook."""
    try:
        test()
    except AssertionError as e:
        print(f"✗ Exercise {n} failed: {e}")
    except Exception as e:
        print(f"✗ Exercise {n} failed: {type(e).__name__}: {e}")
    else:
        print(f"✓ Exercise {n} passed")

**Exercise 1 (the video's exercise).** Write `rule_d(trace)`: return the index of the first entry of the first run of
**three tool errors in a row** (a result that starts with `"ERROR"`), or None. Then build `d_hits`: the sorted list of
`(incident, sample)` keys in `traces` where `rule_d` fires. Check it on our stuck consumer run, where entries 1, 2 and 3
all came back as errors.

In [13]:
def rule_d(trace):
    # YOUR CODE HERE
    pass

d_hits = None   # YOUR CODE HERE

In [14]:
def test1():
    assert rule_d(traces[("stuck_consumer-0", 2)]["trace"]) == 1, "entries 1, 2, 3 are errors: rule d fires at entry 1"
    e = {"tool": "x", "result": "ERROR: nope"}; ok = {"tool": "x", "result": "fine"}
    assert rule_d([e, e, ok, e, e]) is None, "errors with a success between them are not 3 in a row"
    assert rule_d([ok, e, e, e]) == 1
    assert d_hits == [("stuck_consumer-0", 2)], f"over the 39 exact traces only this run has 3 errors in a row, got {d_hits}"
check(1, test1)

✗ Exercise 1 failed: entries 1, 2, 3 are errors: rule d fires at entry 1


**Exercise 2.** Write `count_errors(trace)`: how many entries have a result that starts with `"ERROR"`.

In [15]:
def count_errors(trace):
    # YOUR CODE HERE
    pass

In [16]:
def test2():
    assert count_errors(traces[("stuck_consumer-0", 2)]["trace"]) == 4, "entries 1, 2, 3 and the runbook at 5"
    assert count_errors(traces[("upstream_deploy-0", 0)]["trace"]) == 0
check(2, test2)

✗ Exercise 2 failed: entries 1, 2, 3 and the runbook at 5


**Exercise 3.** Write `tail_after_fix(run)`: how many trace entries come **after** the labelled fix
(use `fix_step`), or None when the fix never happened.

In [17]:
def tail_after_fix(run):
    # YOUR CODE HERE
    pass

In [18]:
def test3():
    assert tail_after_fix(traces[("stuck_consumer-0", 2)]) == 0, "the restart was the last entry"
    assert tail_after_fix(traces[("bad_deploy-2", 0)]) == 5, "bad_deploy-2 s0 kept going 5 entries after the fix"
    assert tail_after_fix(traces[("false_alarm-5", 0)]) is None, "a false alarm has no fix"
check(3, test3)

✗ Exercise 3 failed: the restart was the last entry


**Exercise 4.** Write `repeated_write(row)` for one saved jsonl row: True when the same write (all three parts) appears
twice or more in `row["writes"]`. Count it over all 180 rows.

In [19]:
def repeated_write(row):
    # YOUR CODE HERE
    pass

In [20]:
def test4():
    assert repeated_write({"writes": [["rollback_deploy", "checkout", None]] * 2}) is True, "the same rollback twice is a repeat"
    assert repeated_write({"writes": [["rollback_deploy", "web", None], ["restart_service", "web", None]]}) is False
    n = sum(repeated_write(r) for r in rows)
    assert n == 44, f"expected 44 runs with an identical repeated write, got {n}"
check(4, test4)

✗ Exercise 4 failed: the same rollback twice is a repeat


## Solutions (try first!)

In [21]:
def rule_d(trace):
    run_len = 0
    for i, t in enumerate(trace):
        if str(t.get("result", "")).startswith("ERROR"):
            run_len += 1
            if run_len == 3:
                return i - 2
        else:
            run_len = 0
    return None

d_hits = sorted(k for k, a in traces.items() if rule_d(a["trace"]) is not None)
print(d_hits)

[('stuck_consumer-0', 2)]


In [22]:
def count_errors(trace):
    return sum(str(t.get("result", "")).startswith("ERROR") for t in trace)

In [23]:
def tail_after_fix(run):
    fx = fix_step(run["trace"], run["label"])
    return None if fx is None else len(run["trace"]) - 1 - fx

In [24]:
def repeated_write(row):
    ws = [json.dumps(w) for w in row["writes"]]
    return len(set(ws)) < len(ws)